## ▶ What you'll see when you run this
- Your own text getting **merged into BPE tokens step by step**, then a perfect **encode → decode round-trip** confirmed by `assert`.

**Time:** ~20 min · **Cost:** free (pure Python/NumPy, no model calls) · **Keys:** none

# Week 8 · Notebook 3 — Build a BPE Tokenizer **From Scratch**
**CSCI 250 — Introduction to Artificial Intelligence · Fall 2026**

Every LLM starts by chopping text into **tokens**. Here we build the real algorithm models use — **Byte-Pair Encoding (BPE)** — in plain Python (Karpathy *minbpe* / Stanford CS336 style, simplified).

The plan: start from raw **bytes**, repeatedly **count adjacent pairs** and **merge the most frequent** one into a new token, until we hit a target vocab size. Then we **encode** new text and **decode** it back.

> No API keys, no installs, no GPU. Just the idea behind every tokenizer.

## 0. Wow in 5 seconds — bytes are the starting vocab
Before any merging, a tokenizer sees text as a list of **byte values (0–255)**. That is the base vocabulary every BPE model grows from.

In [ ]:
text = 'the cat sat on the mat. the cat sat.'
tokens = list(text.encode('utf-8'))   # each char -> its byte value
print('characters:', len(text))
print('byte tokens:', tokens[:20], '...')
print('starting vocab size:', 256, '(one entry per possible byte)')

## 1. Count adjacent pairs
BPE looks at every **neighbouring pair** of tokens and counts how often each pair occurs. The most common pair is the best candidate to merge.

In [ ]:
from collections import Counter

def get_pair_counts(ids):
    counts = Counter()
    for a, b in zip(ids, ids[1:]):   # walk neighbouring pairs
        counts[(a, b)] += 1
    return counts

counts = get_pair_counts(tokens)
top = counts.most_common(3)
print('most common adjacent pairs:', top)
# self-check: the most common pair really is the max
assert top[0][1] == max(counts.values())
print('assert passed: top pair is the most frequent ✔')

## 2. Fill-in #1 — merge one pair
**Your turn.** Complete `merge()` so that every occurrence of the pair `(a, b)` in `ids` is replaced by the single new token id `new_id`.

In [ ]:
def merge(ids, pair, new_id):
    a, b = pair
    out = []
    i = 0
    while i < len(ids):
        # TODO: if ids[i], ids[i+1] == a, b -> append new_id and skip 2;
        #       otherwise append ids[i] and skip 1.
        if i < len(ids) - 1 and ids[i] == a and ids[i + 1] == b:
            out.append(new_id)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out

# self-check: merge the pair (6, 7) -> 99
assert merge([5, 6, 7, 6, 7, 8], (6, 7), 99) == [5, 99, 99, 8]
print('assert passed: merge() works ✔')

## 3. Train the tokenizer (repeat: count → merge)
We loop: find the most frequent pair, mint a brand-new token id for it, merge, and remember the rule. Each new token grows the vocabulary by one.

In [ ]:
def train_bpe(text, vocab_size):
    assert vocab_size >= 256
    ids = list(text.encode('utf-8'))
    merges = {}                       # (a, b) -> new_id
    num_merges = vocab_size - 256
    for k in range(num_merges):
        counts = get_pair_counts(ids)
        if not counts:
            break
        pair = max(counts, key=counts.get)   # most frequent pair
        new_id = 256 + k
        ids = merge(ids, pair, new_id)
        merges[pair] = new_id
    return merges, ids

corpus = ('the cat sat on the mat. the cat sat on the hat. '
          'a cat and a hat and a mat. ') * 8
merges, trained_ids = train_bpe(corpus, vocab_size=276)  # 20 merges
print('learned', len(merges), 'merge rules')
print('tokens before:', len(corpus.encode('utf-8')))
print('tokens after :', len(trained_ids))
ratio = len(corpus.encode('utf-8')) / len(trained_ids)
print(f'compression: {ratio:.2f}x fewer tokens')
assert len(trained_ids) < len(corpus.encode('utf-8'))
print('assert passed: BPE shortened the sequence ✔')

## 4. Fill-in #2 — encode new text with the learned merges
**Your turn.** To encode, apply the learned merges **in the order they were learned** (lower new_id first). Complete the loop.

> **Scope cap:** we apply each merge once, in order — enough to see the idea. **Production BPE re-scans the sequence and keeps applying the highest-priority merge until no merge rule applies anymore.** Same algorithm, run to completion.

In [ ]:
def encode(text, merges):
    ids = list(text.encode('utf-8'))
    # apply merges in learned order (new_id ascending)
    for pair, new_id in sorted(merges.items(), key=lambda kv: kv[1]):
        # TODO: replace the pair with new_id using merge()
        ids = merge(ids, pair, new_id)
    return ids

enc = encode('the cat sat on the mat.', merges)
print('encoded:', enc)
# a word the tokenizer saw a lot should be shorter than its raw bytes
assert len(enc) < len('the cat sat on the mat.'.encode('utf-8'))
print('assert passed: encoding compresses familiar text ✔')

## 5. Fill-in #3 — decode back to text (round-trip)
**Your turn.** Build a `vocab` mapping each id to its **bytes**, then decoding is just concatenating those byte-strings. This must round-trip **exactly**.

In [ ]:
def build_vocab(merges):
    vocab = {i: bytes([i]) for i in range(256)}      # base bytes
    for (a, b), new_id in sorted(merges.items(), key=lambda kv: kv[1]):
        vocab[new_id] = vocab[a] + vocab[b]           # a token IS its bytes
    return vocab

def decode(ids, vocab):
    # TODO: join the bytes for each id, then utf-8 decode
    data = b''.join(vocab[i] for i in ids)
    return data.decode('utf-8', errors='replace')

vocab = build_vocab(merges)
sample = 'the cat sat on the mat.'
roundtrip = decode(encode(sample, merges), vocab)
print('original :', sample)
print('roundtrip:', roundtrip)
assert roundtrip == sample, 'round-trip must be exact!'
print('assert passed: encode -> decode is lossless ✔')

## 6. Try it on YOUR text (S-exercise)
Replace the string below with **your own sentence**, then watch it encode and decode. Words your corpus never saw will fall back to raw bytes — that is fine and still round-trips.

In [ ]:
my_text = 'the cat wore a fancy hat'   # <- change me
ids = encode(my_text, merges)
print('your text     :', my_text)
print('your tokens   :', ids)
print('# tokens      :', len(ids), 'vs', len(my_text), 'characters')
assert decode(ids, vocab) == my_text
print('round-trip on YOUR text ✔')

## 7. Vocab size vs sequence length — the core tradeoff
More merges = a **bigger vocabulary** but **shorter sequences** (fewer tokens to process). This is the dial every model maker tunes.

In [ ]:
import numpy as np

sizes = [256, 266, 276, 296, 336]
lengths = []
for v in sizes:
    m, ids = train_bpe(corpus, vocab_size=v)
    lengths.append(len(ids))
for v, L in zip(sizes, lengths):
    print(f'vocab {v:4d} -> {L:4d} tokens')
# bigger vocab never makes the sequence longer
assert all(lengths[i] >= lengths[i + 1] for i in range(len(lengths) - 1))
print('assert passed: bigger vocab -> same-or-shorter sequences ✔')

## 8. Why models can't easily count letters in *strawberry*
A tokenizer often turns a whole word into **one token** — the model literally never sees the individual letters, so 'how many r's in strawberry?' is genuinely hard for it. Tokens also = **API cost**: you pay per token, not per character.

In [ ]:
word = 'strawberry'
# pretend a big-vocab tokenizer mapped this whole word to ONE token:
as_one_token = ['strawberry']
print('the model may see:', as_one_token, '-> 1 opaque token')
print('it does NOT see  :', list(word), '-> the letters are hidden')
print('that is WHY counting letters is unreliable for an LLM.')
# cost intuition: ~4 chars per token, billed per token
prompt = 'Summarize this report in three bullet points. ' * 50
est_tokens = round(len(prompt) / 4)
print(f'\na {len(prompt)}-char prompt ~= {est_tokens} tokens of API cost')

## 9. Compare with a REAL tokenizer (Hugging Face)
Production models use the same BPE idea, just trained on billions of words. The import is guarded so the notebook still runs if `transformers` is missing.

In [ ]:
try:
    from transformers import AutoTokenizer
    hf = AutoTokenizer.from_pretrained('gpt2')   # a real BPE tokenizer
    for s in ['strawberry', 'tokenization', 'the cat sat on the mat']:
        ids = hf.encode(s)
        pieces = [hf.decode([i]) for i in ids]
        print(f'{len(ids):2d} tokens | {s!r} -> {pieces}')
    print('\nSame algorithm you just built, trained at scale.')
except Exception as e:
    print('transformers not available, skipping HF compare:', e)
    print('Your from-scratch BPE above is the real idea either way.')

---
### Recap
bytes → count pairs → merge most frequent → repeat. You built encode + decode that round-trips exactly, saw the **vocab-size vs sequence-length** tradeoff, and connected tokens to **API cost** and the *strawberry* problem. Next notebook: the **attention** math behind the Transformer.